# Шаг 9. Точность на абзацах и оценка статьи целиком

Детектор обучался на длинных текстах, а на сайтах будет видеть отдельные абзацы (обычно 40–220 токенов). Здесь:
1. **Режим «абзац».** Сравниваются три варианта на фрагментах реальной длины: **A** — нынешний классификатор; **B** — отдельные классификаторы для 32, 64, 128 и 256 токенов (расширение выбирает по длине абзаца); **C** — один классификатор, обученный на фрагментах разной длины. Выбор — по валидационной выборке (правило ниже), тест только для отчёта.
2. **Режим «вся статья».** Классификатор по первым 512 токенам объединённого текста статьи.

Языковые модели не запускаются — используются кэши шагов 7–8, поэтому GPU не нужен (подойдёт обычная среда). Время: ≈15–20 минут.
**Прислать:** вывод ячеек 3–5 и файл `results_step9.json`. Файл `weights_v4.json` — скачать себе.

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/ai-text-detector'
os.makedirs('/content/code', exist_ok=True); os.chdir('/content/code'); sys.path.insert(0, '/content/code')

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _convert(word, table):
    return ''.join(table.get(ch, ch) for ch in word)

def _mixed(word):
    return any('\u0400' <= ch <= '\u04FF' for ch in word) and any('a' <= ch.lower() <= 'z' for ch in word)

def _fix_word(word, doc_script):
    """Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно
    (в слове есть буквы без двойников) — к алфавиту большинства букв слова (защита от гомоглифов)."""
    if not _mixed(word):
        # слово целиком из «чужого» алфавита, но только из букв-двойников (например, кириллическая «а» в английском тексте)
        foreign = CYR2LAT if doc_script == 'lat' else LAT2CYR
        if word and all(ch in foreign for ch in word):
            return _convert(word, foreign)
        return word
    first = _convert(word, LAT2CYR if doc_script == 'cyr' else CYR2LAT)
    if not _mixed(first):
        return first
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    return _convert(word, LAT2CYR if cyr > lat else CYR2LAT)

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    doc_script = 'cyr' if cyr_ratio(t) > 0.5 else 'lat'
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0), doc_script), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }

In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res

In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res

In [ ]:
%%writefile /content/code/lm_features.py
"""Вероятностные признаки по компактной авторегрессионной языковой модели (раздел 2.3, формулы 2.1–2.2).
Для каждого токена считаются: логарифм вероятности, энтропия предсказанного распределения, ранг
фактического токена и дисперсия логарифма вероятности (для признака условной кривизны, как в Fast-DetectGPT)."""
import numpy as np

LM_FEATURE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
                    'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature']

def token_stats(texts, model, tok, max_tokens=256, batch_size=8, device='cuda'):
    """Один проход модели по каждому тексту. Возвращает список словарей с массивами по токенам."""
    import torch
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    out = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, max_length=max_tokens,
                  padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)          # прогноз для токенов 2..T
        target = enc.input_ids[:, 1:]
        lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)    # log P(w_i | w_<i)
        p = logp.exp()
        ent = -(p * logp).sum(-1)                                  # энтропия, формула (2.2)
        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)               # дисперсия log p по распределению модели
        rank = (logp > lp.unsqueeze(-1)).sum(-1) + 1               # ранг фактического токена
        mask = enc.attention_mask[:, 1:].bool()
        for b in range(len(batch)):
            m = mask[b]
            out.append({'ids': enc.input_ids[b][enc.attention_mask[b].bool()].cpu().numpy(),
                        'lp': lp[b][m].cpu().numpy(), 'ent': ent[b][m].cpu().numpy(),
                        'var': var[b][m].cpu().numpy(), 'rank': rank[b][m].cpu().numpy()})
        del logits, logp, p
    return out

def aggregate(s, k=None):
    """Признаки по первым k предсказанным токенам (k=None — все)."""
    lp, ent, var, rank = (s[x][:k] if k else s[x] for x in ('lp', 'ent', 'var', 'rank'))
    n = len(lp)
    if n == 0:
        return {f: 0.0 for f in LM_FEATURE_NAMES}
    curv = (lp.sum() + ent.sum()) / np.sqrt(max(var.sum(), 1e-8))   # (∑log p − ∑E[log p]) / √∑Var, E[log p] = −H
    return {
        'mean_lp': float(lp.mean()), 'std_lp': float(lp.std()), 'mean_entropy': float(ent.mean()),
        'mean_log_rank': float(np.log(rank).mean()),
        'rank_1': float((rank == 1).mean()), 'rank_2_10': float(((rank >= 2) & (rank <= 10)).mean()),
        'rank_11_100': float(((rank >= 11) & (rank <= 100)).mean()),
        'rank_101_1000': float(((rank >= 101) & (rank <= 1000)).mean()), 'rank_gt_1000': float((rank > 1000).mean()),
        'curvature': float(curv),
    }


# Дополнительные признаки (шаг 6): вычисляются из тех же величин по токенам, без новых проходов модели.
LM_EXTRA_NAMES = ['lp_q10', 'lp_q50', 'lp_q90', 'std_entropy', 'mean_lp_plus_h',
                  'xent_ratio', 'frac_above_expected', 'mean_var']

def aggregate_ext(s, k=None):
    """lp_q10/50/90 — квантили логарифма вероятности; std_entropy — разброс энтропии;
    mean_lp_plus_h — среднее (log p + H), то есть превышение правдоподобия над ожидаемым в каждой позиции;
    xent_ratio — отношение среднего «удивления» −log p к средней энтропии (одномодельный аналог Binoculars);
    frac_above_expected — доля токенов, для которых log p > −H; mean_var — средняя дисперсия log p."""
    lp, ent, var = (np.asarray(s[x][:k] if k else s[x], dtype=np.float64) for x in ('lp', 'ent', 'var'))
    if len(lp) == 0:
        return {f: 0.0 for f in LM_EXTRA_NAMES}
    return {
        'lp_q10': float(np.quantile(lp, 0.1)), 'lp_q50': float(np.quantile(lp, 0.5)), 'lp_q90': float(np.quantile(lp, 0.9)),
        'std_entropy': float(ent.std()), 'mean_lp_plus_h': float((lp + ent).mean()),
        'xent_ratio': float((-lp).mean() / max(ent.mean(), 1e-6)),
        'frac_above_expected': float((lp > -ent).mean()), 'mean_var': float(var.mean()),
    }

In [ ]:
%%writefile /content/code/hybrid.py
"""Обучение и оценка статистического, стилометрического и гибридного детекторов (разделы 2.3, 3.3)."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from stylometry import stylometric_features, normalize, FEATURE_NAMES as STYLO
from lm_features import aggregate, LM_FEATURE_NAMES as LM
from metrics_utils import evaluate

VARIANTS = {'stylometric': STYLO, 'statistical': LM, 'hybrid': STYLO + LM}

def feature_table(texts, langs, stats, k=None, decode=None):
    """Признаки обеих групп. Если задан k, стилометрия считается по тексту первых k+1 токенов."""
    rows = []
    for t, l, s in zip(texts, langs, stats):
        if k is not None and decode is not None:
            t = decode(s['ids'][:k + 1], l)
        f = stylometric_features(t, l)
        f.update(aggregate(s, k))
        rows.append(f)
    return pd.DataFrame(rows)

def fit(X, y, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)

def threshold_at_fpr(model, X_val, y_val, fpr):
    """Порог, при котором на валидационной выборке доля ложных срабатываний равна fpr."""
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))

def train_eval_all(F_tr, y_tr, F_te, y_te, variants=VARIANTS):
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_tr[cols].values, y_tr)
        out[name] = evaluate(y_te, m.predict_proba(F_te[cols].values)[:, 1])
        models[name] = m
    return out, models

def export(models, thresholds, path, meta):
    """Веса гибридного детектора для расширения."""
    res = {'version': 'hybrid-v1', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        res['langs'][lang] = {'features': STYLO + LM, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]),
                              'thresholds': thresholds[lang]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1)

In [ ]:
%%writefile /content/code/hybrid_v2.py
"""Гибридный детектор, версия 2 (шаг 6): дополнительные вероятностные признаки, подбор регуляризации
на валидационной выборке, обучение на нескольких корпусах. Классификатор — по-прежнему логистическая регрессия,
поэтому результат объясним, а в расширении он вычисляется теми же несколькими арифметическими операциями."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from stylometry import stylometric_features, FEATURE_NAMES as STYLO
from lm_features import aggregate, aggregate_ext, LM_FEATURE_NAMES as LM, LM_EXTRA_NAMES as EXTRA

SETS = {'stylometric': STYLO, 'hybrid_v1': STYLO + LM, 'hybrid_v2': STYLO + LM + EXTRA}
CS = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)


def feature_table(texts, langs, stats):
    rows = []
    for t, l, s in zip(texts, langs, stats):
        f = stylometric_features(t, l); f.update(aggregate(s)); f.update(aggregate_ext(s)); rows.append(f)
    return pd.DataFrame(rows)


def fit(X, y, C=1.0, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)


def select_c(Xtr, ytr, val_sets, wtr=None, Cs=CS):
    """Подбор коэффициента регуляризации C по среднему AUROC на валидационных выборках val_sets = [(X, y), ...]."""
    best = None
    for C in Cs:
        m = fit(Xtr, ytr, C, wtr)
        score = float(np.mean([roc_auc_score(y, m.predict_proba(X)[:, 1]) for X, y in val_sets]))
        if best is None or score > best[1]:
            best = (C, score, m)
    return best   # (C, средний AUROC на валидации, модель)


def corpus_weights(groups):
    """Веса примеров: суммарный вес каждого корпуса одинаков."""
    groups = np.asarray(groups); w = np.ones(len(groups))
    for g in np.unique(groups):
        w[groups == g] = len(groups) / (len(np.unique(groups)) * (groups == g).sum())
    return w


def threshold_at_fpr(model, X_val, y_val, fpr):
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))


def model_json(model, cols, thresholds, C):
    sc, lr = model[0], model[-1]
    return {'features': list(cols), 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
            'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]), 'thresholds': thresholds, 'C': C}


def export(langs, path, meta):
    """langs = {lang: {'main': dict, 'stage1': dict}} — основной классификатор и быстрый стилометрический (первая ступень)."""
    res = {'version': 'hybrid-v2', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, d in langs.items():
        res['langs'][lang] = dict(d['main']); res['langs'][lang]['stage1'] = d['stage1']
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1, ensure_ascii=False)

In [ ]:
%%writefile /content/code/length_models.py
"""Шаг 9. Классификаторы под длину текста (режим «абзац») и классификатор для оценки статьи целиком (512 токенов).
Признаки фрагмента длиной L токенов: вероятностные — по первым L токенам (модель причинная, поэтому они совпадают
с теми, что получились бы при подаче только этих токенов), стилометрические — по тексту, восстановленному из них."""
import numpy as np, pandas as pd
from stylometry import stylometric_features
from lm_features import aggregate, aggregate_ext

BINS = [(48, 32), (96, 64), (192, 128), (10 ** 9, 256)]     # (до скольких токенов включительно, длина обучения)


def bin_for(n_tokens):
    for upto, L in BINS:
        if n_tokens <= upto:
            return L
    return BINS[-1][1]


def row_at(text, lang, s, L, decode, full_text_stylometry=False):
    """Признаки фрагмента: первые L токенов. full_text_stylometry=True — стилометрия по всему тексту (как раньше)."""
    ids = s['ids'][:L]
    t = text if full_text_stylometry else decode([int(i) for i in ids], lang)
    f = stylometric_features(t, lang)
    f.update(aggregate(s, L - 1)); f.update(aggregate_ext(s, L - 1))
    f['_n_tokens'] = int(min(L, len(s['ids'])))
    return f


def table_at(texts, langs, stats, L, decode, full_text_stylometry=False):
    return pd.DataFrame([row_at(t, l, s, L, decode, full_text_stylometry) for t, l, s in zip(texts, langs, stats)])


def table_random_lengths(texts, langs, stats, decode, lo=40, hi=220, seed=0):
    """Каждому тексту — своя случайная длина «абзаца» от lo до hi токенов (типичный диапазон абзаца на сайте)."""
    rng = np.random.default_rng(seed)
    Ls = rng.integers(lo, hi + 1, len(texts))
    return pd.DataFrame([row_at(t, l, s, int(L), decode) for t, l, s, L in zip(texts, langs, stats, Ls)])


def predict_by_length(models, F, cols):
    """models = {L: модель}; для каждой строки выбирается модель по числу токенов фрагмента."""
    p = np.zeros(len(F))
    bins = F['_n_tokens'].map(bin_for).values
    for L, m in models.items():
        idx = np.where(bins == L)[0]
        if len(idx):
            p[idx] = m.predict_proba(F.iloc[idx][cols].values)[:, 1]
    return p

## 3. Данные, кэш признаков (512 токенов, правая обрезка), нынешние классификаторы

In [ ]:
import importlib, pickle, json, numpy as np, pandas as pd
import stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, hybrid_v2, length_models
for m in (stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, hybrid_v2, length_models): importlib.reload(m)
from stylometry import normalize
from baseline_stylometry import split
from metrics_utils import evaluate
from hybrid_v2 import SETS, fit, select_c, threshold_at_fpr, model_json
from length_models import table_at, table_random_lengths, predict_by_length, BINS
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer
SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet'); df = df[df.dataset == 'M4']
def bal(d, cap): return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {'en': bal(df[df.lang == 'en'], 6000), 'ru': df[df.lang == 'ru']}
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
S = pickle.load(open(f'{WORK}/lm_stats_512.pkl', 'rb'))
assert all(len(S[l]) == len(data[l]) for l in ['en', 'ru']), 'Кэш не совпадает с данными — напиши мне'
TOK = {'en': AutoTokenizer.from_pretrained('gpt2'), 'ru': AutoTokenizer.from_pretrained('ai-forever/rugpt3small_based_on_gpt2')}
decode = lambda ids, lang: TOK[lang].decode(ids)
W = json.load(open(f'{WORK}/weights_v3.json', encoding='utf-8'))
COLS = {l: W['langs'][l]['features'] for l in ['en', 'ru']}
CMAIN = {l: W['langs'][l].get('C', 1.0) for l in ['en', 'ru']}
print('Наборы признаков в расширении:', {l: len(c) for l, c in COLS.items()}, '| C:', CMAIN)
SPL, Y = {}, {}
for l in ['en', 'ru']:
    tr, va, te = split(data[l], SEED); SPL[l] = {'tr': tr.index.values, 'va': va.index.values, 'te': te.index.values}
    Y[l] = data[l].label.values.astype(int)
def part(l, p, F): idx = SPL[l][p]; return F.iloc[idx], Y[l][idx]
RES = {}
def save(): json.dump(RES, open(f'{WORK}/results_step9.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
FT = {}
for l in ['en', 'ru']:
    texts, st = list(data[l].norm), S[l]; langs = [l] * len(texts)
    FT[l] = {'full': table_at(texts, langs, st, 256, decode, full_text_stylometry=True)}   # как обучен нынешний классификатор
    for L in [32, 64, 128, 256, 512]: FT[l][L] = table_at(texts, langs, st, L, decode)
    FT[l]['rand'] = table_random_lengths(texts, langs, st, decode, seed=SEED)
    print(l, 'признаки посчитаны; средняя длина «абзаца» в случайной выборке:', round(FT[l]['rand']._n_tokens.mean()), 'токенов')

## 4. Режим «абзац»: A, B, C
**Правило выбора (задано до запуска):** вариант с наибольшим AUROC на валидационной выборке со случайной длиной фрагментов (40–220 токенов). Пороги подсветки для B подбираются отдельно для каждой длины.

In [ ]:
RES['paragraph'] = {}; CHOICE = {}
for l in ['en', 'ru']:
    cols = COLS[l]; out = {}
    Xa, ya = part(l, 'tr', FT[l]['full']); A = fit(Xa[cols].values, ya, C=CMAIN[l])
    B = {}
    for L in [32, 64, 128, 256]:
        Xtr, ytr = part(l, 'tr', FT[l][L]); Xva, yva = part(l, 'va', FT[l][L])
        B[L] = select_c(Xtr[cols].values, ytr, [(Xva[cols].values, yva)])[2]
    Xc = pd.concat([part(l, 'tr', FT[l][L])[0] for L in [32, 64, 128, 256]]); yc = np.concatenate([part(l, 'tr', FT[l][L])[1] for L in [32, 64, 128, 256]])
    Xcv = pd.concat([part(l, 'va', FT[l][L])[0] for L in [32, 64, 128, 256]]); ycv = np.concatenate([part(l, 'va', FT[l][L])[1] for L in [32, 64, 128, 256]])
    Cc = select_c(Xc[cols].values, yc, [(Xcv[cols].values, ycv)])[2]
    preds = {'A: нынешний': lambda F: A.predict_proba(F[cols].values)[:, 1],
             'B: по длине': lambda F: predict_by_length(B, F, cols),
             'C: смешанные длины': lambda F: Cc.predict_proba(F[cols].values)[:, 1]}
    for name, f in preds.items():
        Fv, yv = part(l, 'va', FT[l]['rand']); Ft, yt = part(l, 'te', FT[l]['rand'])
        r = {'val_rand': float(roc_auc_score(yv, f(Fv))), 'test_rand': evaluate(yt, f(Ft))}
        for L in [32, 64, 128, 256]:
            FtL, ytL = part(l, 'te', FT[l][L]); r[f'test_{L}'] = float(roc_auc_score(ytL, f(FtL)))
        FtF, ytF = part(l, 'te', FT[l]['full']); r['test_full_text'] = float(roc_auc_score(ytF, f(FtF)))
        out[name] = r
        print(f"{l} | {name:20s} вал.(случ. длина)={r['val_rand']:.3f} | тест (случ. длина): AUROC={r['test_rand']['AUROC']:.3f} "
              f"[{r['test_rand']['AUROC_95CI'][0]:.3f}; {r['test_rand']['AUROC_95CI'][1]:.3f}] TPR@1%={r['test_rand']['TPR@1%FPR']:.3f} | "
              + ' '.join(f"{L}:{r[f'test_{L}']:.3f}" for L in [32, 64, 128, 256]) + f" | длинный текст: {r['test_full_text']:.3f}")
    best = max(out, key=lambda n: out[n]['val_rand'])
    CHOICE[l] = (best, A, B, Cc); out['chosen'] = best; RES['paragraph'][l] = out; save()
    print(f'>>> {l}: выбран «{best}»\n')

## 5. Режим «вся статья» (первые 512 токенов) и сборка весов для расширения

In [ ]:
RES['article'] = {}
for l in ['en', 'ru']:
    cols = COLS[l]
    Xtr, ytr = part(l, 'tr', FT[l][512]); Xva, yva = part(l, 'va', FT[l][512]); Xte, yte = part(l, 'te', FT[l][512])
    C, _, m = select_c(Xtr[cols].values, ytr, [(Xva[cols].values, yva)])
    r = evaluate(yte, m.predict_proba(Xte[cols].values)[:, 1])
    thr = {'fpr10': threshold_at_fpr(m, Xva[cols].values, yva, 0.10), 'fpr02': threshold_at_fpr(m, Xva[cols].values, yva, 0.02)}
    W['langs'][l]['article'] = {**model_json(m, cols, thr, C), 'max_tokens': 512}
    RES['article'][l] = {'C': C, 'test': r, 'thresholds': thr}
    print(f"{l}: статья целиком (512 токенов): AUROC={r['AUROC']:.3f} [{r['AUROC_95CI'][0]:.3f}; {r['AUROC_95CI'][1]:.3f}] "
          f"F1={r['F1']:.3f} FPR={r['FPR']:.3f} TPR@1%={r['TPR@1%FPR']:.3f}")
    best, A, B, Cc = CHOICE[l]
    if best.startswith('B'):
        entries = []
        for upto, L in BINS:
            Xv, yv = part(l, 'va', FT[l][L])
            thrL = {'fpr10': threshold_at_fpr(B[L], Xv[cols].values, yv, 0.10), 'fpr02': threshold_at_fpr(B[L], Xv[cols].values, yv, 0.02)}
            entries.append({**model_json(B[L], cols, thrL, None), 'upto_tokens': upto, 'trained_tokens': L})
        W['langs'][l]['by_length'] = entries
    elif best.startswith('C'):
        Xcv = pd.concat([part(l, 'va', FT[l][L])[0] for L in [32, 64, 128, 256]]); ycv = np.concatenate([part(l, 'va', FT[l][L])[1] for L in [32, 64, 128, 256]])
        thrC = {'fpr10': threshold_at_fpr(Cc, Xcv[cols].values, ycv, 0.10), 'fpr02': threshold_at_fpr(Cc, Xcv[cols].values, ycv, 0.02)}
        stage1 = W['langs'][l]['stage1']; W['langs'][l] = {**model_json(Cc, cols, thrC, None), 'stage1': stage1, 'article': W['langs'][l]['article']}
W['version'] = 'hybrid-v4'; W['meta']['paragraph_mode'] = {l: CHOICE[l][0] for l in CHOICE}
json.dump(W, open(f'{WORK}/weights_v4.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
save(); print('Сохранено: weights_v4.json, results_step9.json')